# Lab 8 - Financial Analyst with the `xlsx` Skill

**Section 10 - Skills**  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

## Goal
Build a financial-analyst agent that turns a CSV of monthly revenue into a polished Excel workbook: formatted sheets, live formulas, totals, and charts. The work is done by attaching the prebuilt **`xlsx`** skill, which loads reactively the moment the task is about spreadsheets, so your system prompt stays lean.

This lab creates its **own dedicated data-science environment** with pandas, numpy, and matplotlib. That keeps Lab 8 independent: you can run it without first completing Lab 4 or copying an environment id between notebooks.

By the end you will have a real `.xlsx` on disk that opens in Excel with a chart and working formulas, retrieved from the session via the Files API.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`.
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- A Managed Agents cloud environment will be created by this notebook with pandas, numpy, and matplotlib.
- **The prebuilt `xlsx` skill** (an Anthropic-maintained office skill). You attach it by `skill_id`; nothing to install.
- The sample **`revenue.csv`** lives in this lab folder, next to the notebook, so the upload cell finds it with a relative path.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From labs/code, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()
print("ANTHROPIC_API_KEY configured for this notebook session.")
print("Lab 8 will create its own data-science environment below.")

In [ ]:
import os
from pathlib import Path
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Create a dedicated data-science environment
Create a Lab 8-specific cloud environment with pandas, numpy, and matplotlib. This avoids a hard dependency on Lab 4 while keeping the same analysis stack.

In [ ]:
env = client.beta.environments.create(
    name="financial-analyst-data",
    config={
        "type": "cloud",
        "packages": {
            "pip": ["pandas==2.2.0", "numpy", "matplotlib"],
        },
        "networking": {"type": "unrestricted"},
    },
    betas=BETAS,
)
print("env.id =", env.id)

### Step 2 - Attach the prebuilt `xlsx` skill to the agent
Create the agent with the full toolset and attach the `xlsx` skill by ID. The skill is loaded on demand, so it costs nothing on turns that are not about spreadsheets. Remember the ceiling is 20 skills per session, counted across agents.

In [ ]:
agent = client.beta.agents.create(
    name="Financial Analyst",
    model=MODEL,
    system=(
        "You are a financial analyst. Produce clean, professional Excel "
        "workbooks. Always use the xlsx skill for spreadsheet output. "
        "Place all deliverables in /mnt/session/outputs/."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    skills=[{"type": "anthropic", "skill_id": "xlsx"}],
    betas=BETAS,
)
print("agent.id =", agent.id)

### Step 3 - Upload and mount `revenue.csv`
Upload the CSV from this lab folder, then mount it into the session at a known path so the agent can read it.

In [ ]:
csv = client.beta.files.upload(file=Path("revenue.csv"), betas=BETAS)
print("file.id  =", csv.id)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    resources=[{
        "type": "file",
        "file_id": csv.id,
        "mount_path": "/workspace/revenue.csv",
    }],
    title="Revenue summary workbook",
    betas=BETAS,
)
print("session.id =", session.id)

### Step 4 - Ask the agent to analyze the CSV and produce a formatted `.xlsx`
Be explicit about the sheets, formulas, charts, and the output path. The agent loads the `xlsx` skill the moment it sees an Excel request and writes the file to `/mnt/session/outputs/`.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "Analyze /workspace/revenue.csv and build a polished Excel "
                "workbook. Include a 'Summary' sheet with monthly revenue, "
                "cost, margin, and margin % columns, a 'Totals' row, and "
                "month-over-month growth %. Add a column chart of revenue "
                "by month and a line chart of margin %. Format headers, "
                "currency, and percentages cleanly. Save the workbook to "
                "/mnt/session/outputs/revenue_summary.xlsx."
            ),
        }],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.tool_use":
            print(f"\n[tool: {event.name}]")
        elif event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Step 5 - Retrieve the workbook via the Files API
List the files the session produced and download the `.xlsx` locally. Filter on `.xlsx` so you skip the mounted input CSV.

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
for f in client.beta.files.list(scope_id=session.id, betas=BETAS):
    print(f.id, f.filename)
    if f.filename.endswith(".xlsx"):
        client.beta.files.download(f.id).write_to_file(str(out_dir / f.filename))
        print("saved:", out_dir / f.filename)

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- The agent loads the `xlsx` skill reactively once the task mentions Excel.
- Tool calls stream by: `read` the CSV, `bash` to build the workbook, `write` the output.
- A polished `revenue_summary.xlsx` lands in `/mnt/session/outputs/` with: formatted headers, revenue / cost / margin / margin % columns, a Totals row, month-over-month growth, and a column chart plus a line chart.
- The workbook downloads into your local `outputs/` folder and opens cleanly in Excel with live formulas and charts.
- Session status moves `running` to `idle`.

## Troubleshooting
- **Skill not loaded / no Excel output** -> confirm `skills=[{"type": "anthropic", "skill_id": "xlsx"}]` is on the agent. The skill only loads when the task reads as a spreadsheet request, so make the prompt clearly about an `.xlsx` workbook.
- **The 20-skill ceiling** -> a session may attach at most **20 skills**, counted across all agents in the session. This lab uses one (`xlsx`); if you also add `docx` or `pptx` in the stretch, stay well under the limit and only attach what the agent actually uses.
- **Environment creation fails** -> confirm your Anthropic API key has Managed Agents access and rerun the environment cell. This lab creates its own `financial-analyst-data` environment, so no `ENV_ID` from Lab 4 is required.
- **File not in outputs** -> only files written to the exact path `/mnt/session/outputs/` are collected by the Files API. If nothing downloads, tell the agent explicitly to save to that directory.
- **Retrieval returns nothing** -> list with the same `betas` and use `scope_id=session.id`. Filter on `.xlsx` so you skip the mounted input CSV.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste the prompts in order.

**Prompt 1 - build and run the analyst with its own environment:**

> "Create a Managed Agents cloud environment named `financial-analyst-data` that pre-installs pandas pinned to 2.2.0, numpy, and matplotlib, with unrestricted networking. Then create a Managed Agents agent named `Financial Analyst` on `claude-haiku-4-5-20251001` with the full agent toolset and the prebuilt **`xlsx`** skill attached (`skill_id` `xlsx`). System prompt: it is a financial analyst that always uses the xlsx skill and writes deliverables to `/mnt/session/outputs/`. Upload `revenue.csv` and mount it at `/workspace/revenue.csv`. Start a session on that environment, then ask the agent to analyze the CSV and build a polished `.xlsx` with a Summary sheet (revenue, cost, margin, margin %, a Totals row, month-over-month growth %), a column chart of revenue, and a line chart of margin %, saved to `/mnt/session/outputs/revenue_summary.xlsx`. Stream the response."

**Prompt 2 - retrieve the file:**

> "List the files this session produced and download the `.xlsx` into a local `outputs/` folder."

**Prompt 3 (optional) - sharpen the deliverable:**

> "If the workbook is missing a chart or the formulas are static values, send a follow-up asking the agent to add a column chart of revenue by month and use live Excel formulas for the Totals row."

## Stretch
- **Add a Word summary.** Attach the `docx` skill alongside `xlsx` and ask for a one-page `.docx` executive summary of the workbook (headline numbers, trend, one chart image). Two office skills, still well under the 20-skill ceiling.
- **Pivot tables.** Ask the agent to add a pivot-table sheet that breaks revenue and margin out by quarter, with a quarter-over-quarter comparison.
- **Generalize it.** Wrap the flow in a function that takes any CSV path and output name, then run it against a second CSV to confirm it generalizes without code changes.

## What you've learned
- How to attach an Anthropic prebuilt skill (`xlsx`) by `skill_id` and rely on reactive, on-demand loading to keep system prompts lean.
- That skills produce real binary artifacts: a formatted Excel workbook, not just text.
- How to create a dedicated cloud environment for a skill-backed workflow.
- The upload -> mount -> outputs -> Files API flow for working with user files end to end.